In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('/kaggle/input/System-Threat-Forecaster/train.csv')

In [ ]:
df=pd.DataFrame(df)
df.dropna()
X = df.drop(columns=['target'])  # Features
y = df['target']  # Target variable

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin
from lightgbm import LGBMClassifier

# Function to detect column types
def detect_column_types(X):
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    return numerical_cols, categorical_cols

# Custom Transformer to Drop Redundant Columns
class DropRedundantColumns(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        # Identify columns with a single unique value
        self.redundant_columns_ = [col for col in X.columns if X[col].nunique() == 1]
        return self

    def transform(self, X):
        return X.drop(columns=self.redundant_columns_, errors='ignore')

# Preprocessing for numerical features
num_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))  # LightGBM handles unscaled values well
])

# Preprocessing for categorical features
cat_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))  # Keeps unknown categories in test data
])

# Full preprocessing pipeline
full_pipeline = Pipeline(steps=[
    ('drop_redundant', DropRedundantColumns()),
    ('column_transformer', 'passthrough')  # Placeholder to update later
])

# Fit and transform to detect column types
full_pipeline.fit(X)
df_reduced = full_pipeline.named_steps['drop_redundant'].transform(X)

# Detect column types
numerical_cols, categorical_cols = detect_column_types(df_reduced)

# Update the preprocessing step with correct column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_preprocessor, numerical_cols),
        ('cat', cat_preprocessor, categorical_cols)
    ]
)

# Update final pipeline
full_pipeline.steps[-1] = ('column_transformer', preprocessor)

# Apply preprocessing to the dataset
X_preprocessed = full_pipeline.fit_transform(X, y)

# Convert categorical columns to integer (for LightGBM's native handling)
for col in categorical_cols:
    X[col] = X[col].astype('category')

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.metrics import accuracy_score


# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, random_state=16)

# Objective function for Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True), 
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0), 
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True), 
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True), 
        'boosting_type': 'gbdt',
        'objective': 'binary', 
        'random_state': 42,
        'verbosity': -1
    }
    
    model = lgb.LGBMClassifier(**params, n_jobs=-1)
    score = cross_val_score(model, X_preprocessed, y, cv=3, scoring='accuracy').mean()
    
    return score

# Run Optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50,n_jobs=-1)

# Best Hyperparameters
print("Best Hyperparameters:", study.best_params)

# Train Final Model with Best Params
best_params = study.best_params
best_model = lgb.LGBMClassifier(**best_params, random_state=42, n_jobs=-1)
best_model.fit(X_preprocessed, y)

# Evaluate Accuracy
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Optimized LightGBM Accuracy: {accuracy * 100:.2f}%')


# Load External Test Data
try:
    X_testf = pd.read_csv('/kaggle/input/System-Threat-Forecaster/test.csv')
    print(f'Test file loaded successfully. Shape: {X_testf.shape}')
except FileNotFoundError:
    print("Error: Test file not found. Check the path.")
    exit()

# Apply the same preprocessing to external test data
X_testf_transformed = full_pipeline.transform(X_testf)

# Make Predictions
y_testf = best_model.predict(X_testf_transformed)
submission = pd.DataFrame({"id": range(0, X_testf.shape[0]), "target": y_testf})

submission.to_csv('submission.csv', index=False)
print(f'Submission file saved. Shape: {submission.shape}')